# Exploracion de Planillas Fuente (.xlsm)

Las planillas originales por instrumento y año son archivos `.xlsm` multi-hoja con macros,
entregados por la contraparte (VcM). Su estructura es distinta a los exports planos (`.xlsx`)
que procesa el pipeline actual.

**Objetivo**: caracterizar su estructura (hojas, columnas, familias de formato) antes de
decidir como integrarlas al pipeline. Foco en localizar datos de actividades
planificadas/ejecutadas para el KPI I19.

---

In [1]:
import sys
from pathlib import Path
import warnings

_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / ".."]
PROJECT_ROOT = next((p.resolve() for p in _candidates if (p / "src" / "__init__.py").exists()), Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
sns.set_theme(style="whitegrid", font_scale=0.9)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 60)

In [2]:
DATA_DIR = Path("/Users/ignacioramirez/Desktop/Proyecto VCM/drive-download-20260623T174348Z-3-001/iniciativas_a_excel/")
XLSM_FILES = sorted(DATA_DIR.glob("*.xlsm"))
print(f"Total archivos .xlsm encontrados: {len(XLSM_FILES)}")
for f in XLSM_FILES:
    print(f"  - {f.name}")

Total archivos .xlsm encontrados: 0


---
## 1. Inventario de hojas por archivo

Para cada `.xlsm`: nombre de hoja, dimensiones, y proposito aparente.

In [3]:
inventario = []

for f in XLSM_FILES:
    xl = pd.ExcelFile(f, engine="openpyxl")
    for sheet in xl.sheet_names:
        df_s = xl.parse(sheet)
        inventario.append({
            "archivo": f.name,
            "hoja": sheet,
            "filas": df_s.shape[0],
            "columnas": df_s.shape[1],
        })

df_inv = pd.DataFrame(inventario)

# Proposito aparente por nombre de hoja
def proposito(hoja):
    h = hoja.upper().strip()
    if h in ("PLANILLA", "PLANILLA "):
        return "Iniciativas (datos de cada proyecto)"
    if "ACTIVIDAD" in h:
        return "Matriz carrera x actividad (pivot)"
    if h == "MATRIZ":
        return "Listas de valores / resumen categorico"
    if h == "CONTRIBUCIONES":
        return "Catalogo actividad-contribucion"
    if h.startswith("HOJA"):
        return "Hoja auxiliar / vacia"
    return "Otro"

df_inv["proposito"] = df_inv["hoja"].apply(proposito)
print(df_inv.to_string(index=False))

KeyError: 'hoja'

---
## 2. Familias de formato

Agrupamos los archivos por su firma de hojas y estructura de columnas.
Hipotesis: hay al menos 3 familias (2022-2023, 2024, 2024-2025 centralizadas).

In [ ]:
# Firma = tuple de nombres de hoja normalizados (sin Hoja1/Hoja3 auxiliares)
def firma_hojas(archivo):
    hojas = df_inv[df_inv["archivo"] == archivo]["hoja"].tolist()
    # Excluir hojas auxiliares vacias
    hojas_norm = sorted([h for h in hojas if not h.startswith("Hoja")])
    return tuple(hojas_norm)

firmas = {}
for f in XLSM_FILES:
    firma = firma_hojas(f.name)
    firmas.setdefault(firma, []).append(f.name)

print(f"Familias de formato por estructura de hojas: {len(firmas)}")
print()
for i, (firma, archivos) in enumerate(firmas.items(), 1):
    print(f"Familia {i}: hojas = {list(firma)}")
    for a in archivos:
        print(f"  - {a}")
    print()

### 2.1 Refinamiento: familias por columnas de la hoja Planilla

Dentro de cada familia de hojas, las columnas de la hoja principal (Planilla/PLANILLA)
pueden variar. Agrupamos por el set de columnas real.

In [ ]:
# Leer columnas de la hoja principal de cada archivo
planilla_cols = {}
for f in XLSM_FILES:
    xl = pd.ExcelFile(f, engine="openpyxl")
    # Buscar hoja principal
    hoja_p = None
    for s in xl.sheet_names:
        if s.upper().strip() == "PLANILLA":
            hoja_p = s
            break
    if hoja_p:
        df_p = xl.parse(hoja_p, nrows=0)
        planilla_cols[f.name] = list(df_p.columns)

# Agrupar por set de columnas
from collections import defaultdict
familias_cols = defaultdict(list)
for archivo, cols in planilla_cols.items():
    key = tuple(cols)
    familias_cols[key].append(archivo)

print(f"Familias de formato por columnas de Planilla: {len(familias_cols)}")
print()
familia_nombres = {}
for i, (cols, archivos) in enumerate(sorted(familias_cols.items(), key=lambda x: len(x[1]), reverse=True), 1):
    nombre_familia = f"F{i}"
    familia_nombres[cols] = nombre_familia
    print(f"{nombre_familia} ({len(cols)} columnas, {len(archivos)} archivos):")
    for a in archivos:
        print(f"  - {a}")
    print(f"  Columnas: {list(cols)[:10]}{'...' if len(cols)>10 else ''}")
    print()

---
## 3. Tabla maestra de columnas por familia

Muestra que columna existe en que familia de formato, para visualizar la evolucion del esquema.

In [ ]:
# Construir union de todas las columnas
todas_cols = set()
for cols in familias_cols.keys():
    todas_cols.update(cols)

# Tabla maestra: columna x familia
familias_keys = sorted(familias_cols.keys(), key=lambda x: len(familias_cols[x]), reverse=True)
familia_labels = [familia_nombres[k] for k in familias_keys]

# Ordernar columnas por primera aparicion
col_orden = []
for k in familias_keys:
    for c in k:
        if c not in col_orden:
            col_orden.append(c)

tabla_maestra = pd.DataFrame(index=col_orden, columns=familia_labels)
for k, label in zip(familias_keys, familia_labels):
    for c in col_orden:
        tabla_maestra.at[c, label] = "X" if c in k else ""

print("Tabla maestra: columna presente (X) por familia de formato")
print(f"({len(col_orden)} columnas unicas en total)")
print()
print(tabla_maestra.to_string())

In [ ]:
# Visualizacion: heatmap de presencia de columnas
tabla_bin = tabla_maestra.replace({"X": 1, "": 0}).astype(int)

fig, ax = plt.subplots(figsize=(4, max(8, len(col_orden) * 0.3)))
sns.heatmap(tabla_bin, cmap="Blues", cbar=False, linewidths=0.5, ax=ax,
            xticklabels=familia_labels, yticklabels=col_orden)
ax.set_title("Presencia de columnas por familia de formato", fontsize=11)
ax.set_xlabel("Familia")
plt.tight_layout()
plt.show()

---
## 4. Foco en KPI I19: actividades planificadas vs ejecutadas

El indicador I19 requiere contrastar actividades planificadas con actividades
efectivamente ejecutadas/realizadas. Buscamos en que hoja y columna esta ese dato.

In [ ]:
# Buscar columnas de actividades planificadas/ejecutadas (excluir "Financiamiento Ejecutado")
i19_hallazgos = []

for f in XLSM_FILES:
    xl = pd.ExcelFile(f, engine="openpyxl")
    for sheet in xl.sheet_names:
        if sheet.startswith("Hoja"):
            continue
        df_s = xl.parse(sheet, nrows=0)
        cols_interes = [c for c in df_s.columns if any(
            kw in str(c).lower() for kw in ["act planificad", "act eje", "act realizad"]
        )]
        if cols_interes:
            i19_hallazgos.append({
                "archivo": f.name,
                "hoja": sheet,
                "columnas": cols_interes,
            })

print(f"Hallazgos de columnas Planificadas/Ejecutadas: {len(i19_hallazgos)}")
print()
for h in i19_hallazgos:
    print(f"  {h['archivo'][:55]}")
    print(f"    Hoja: {h['hoja']} -> {h['columnas']}")
    print()

### 4.1 Detalle: datos reales de act. planificadas vs ejecutadas

Mostramos una muestra de los datos donde ambas columnas existen
("Cantidad Act Planificadas" y "Cantidad de Act Ejecutadas").

In [ ]:
# Archivos que tienen AMBAS columnas (planificadas y ejecutadas)
ambas = [h for h in i19_hallazgos if len(h["columnas"]) >= 2]
print(f"Archivos con AMBAS columnas (planificadas + ejecutadas): {len(ambas)}")
print()

# Solo planificadas (sin ejecutadas)
solo_plan = [h for h in i19_hallazgos if len(h["columnas"]) == 1]
print(f"Archivos con SOLO planificadas (sin ejecutadas): {len(solo_plan)}")
for h in solo_plan:
    print(f"  - {h['archivo'][:60]} -> {h['columnas']}")
print()

# Muestra de datos donde hay ambas
if ambas:
    ejemplo = ambas[0]
    df_ej = pd.read_excel(DATA_DIR / ejemplo["archivo"], sheet_name=ejemplo["hoja"], engine="openpyxl")
    cols_mostrar = ["Codigo "] + ejemplo["columnas"] if "Codigo " in df_ej.columns else ejemplo["columnas"]
    cols_mostrar = [c for c in cols_mostrar if c in df_ej.columns]
    print(f"Muestra de {ejemplo['archivo']} [{ejemplo['hoja']}]:")
    print(df_ej[cols_mostrar].dropna(how="all").head(10).to_string(index=False))

### 4.2 Cobertura temporal del dato de actividades ejecutadas

El dato critico para I19 es "actividades ejecutadas". Veamos en que años esta disponible.

In [ ]:
# Resumen: por archivo, existe planificadas? existe ejecutadas? cuantas filas con dato?
resumen_i19 = []
for f in XLSM_FILES:
    xl = pd.ExcelFile(f, engine="openpyxl")
    # Buscar hoja principal
    hoja_p = None
    for s in xl.sheet_names:
        if s.upper().strip() == "PLANILLA":
            hoja_p = s
            break
    if not hoja_p:
        continue
    df_p = xl.parse(hoja_p)
    
    col_plan = next((c for c in df_p.columns if "act planificad" in str(c).lower()), None)
    col_ejec = next((c for c in df_p.columns if "act eje" in str(c).lower() or "act realizad" in str(c).lower()), None)
    
    n_plan = df_p[col_plan].notna().sum() if col_plan else 0
    n_ejec = df_p[col_ejec].notna().sum() if col_ejec else 0
    
    resumen_i19.append({
        "archivo": f.name[:50],
        "col_planificadas": col_plan or "-",
        "n_plan": n_plan,
        "col_ejecutadas": col_ejec or "-",
        "n_ejec": n_ejec,
        "total_filas": len(df_p),
    })

df_i19 = pd.DataFrame(resumen_i19)
print("Disponibilidad de Act. Planificadas y Ejecutadas por archivo:")
print(df_i19.to_string(index=False))
print()
print(f"Archivos con dato de ejecutadas: {(df_i19['n_ejec'] > 0).sum()} de {len(df_i19)}")
print(f"Archivos SIN dato de ejecutadas: {(df_i19['n_ejec'] == 0).sum()} de {len(df_i19)}")

---
## 5. Resumen y recomendaciones

### Familias de formato identificadas

Se identifican **3 familias principales** de formato en las planillas fuente:

| Familia | Periodo | Hojas | Cols Planilla | Archivos | Diferencia clave |
|---------|---------|-------|---------------|----------|------------------|
| F-Legacy | 2022-2023 | PLANILLA, ACTIVIDAD, Matriz | 28-30 | 6 (EXT/FCR/VEDP 2022-2023) | Nombres cortos, tiene Act Ejecutadas, nomenclatura antigua |
| F-2024 | 2024 | Planilla, Matriz Actividad, Matriz, Contribuciones | 34-44 | 3-5 (EXT/VEDP/UTG 2024, Centralizadas) | Nombres alineados con export SISAV2, sin Act Ejecutadas |
| F-VEDP24 | 2024 (VEDP) | Planilla, Matriz Actividad, Contribuciones, Matriz | 44 | 1 (VEDP 2024) | Incluye campos curriculares (catedra, logros) |

### Hallazgos sobre KPI I19

- **"Cantidad de Act Ejecutadas"** existe SOLO en la familia 2022-2023 (6 archivos).
- Los archivos 2024-2025 tienen "Cantidad Act Planificadas" pero NO "Act Ejecutadas".
- Esto significa que el dato de cumplimiento (ejecutadas/planificadas) solo es computable
  para el periodo 2022-2023 con las planillas actuales.
- La hoja ACTIVIDAD/Matriz Actividad NO contiene datos de actividades individuales;
  es una tabla pivote de carreras (filas) por carreras (columnas), sin utilidad para I19.

### Mapeos de esquema necesarios

Para absorber estas planillas en el pipeline se necesitarian:

1. **Mapeo F-Legacy -> canonico**: renombrar ~20 columnas (nomenclatura antigua a nueva).
   Requiere decision sobre campos que no existen en el canonico actual (Financiamiento,
   Financiamiento Ejecutado, Estado de Avance, Total Asistentes, Estrategias desarrollo).
2. **Mapeo F-2024 -> canonico**: minimo (ya estan casi alineados con el export plano).
3. **Mapeo F-VEDP24 -> canonico**: extension del anterior con campos curriculares adicionales.
4. **Extraccion de Act Ejecutadas** de F-Legacy como columna adicional (no existe en exports
   planos ni en formatos 2024+).

**Recomendacion**: priorizar la integracion de F-Legacy (2022-2023) porque es la unica fuente
del dato de actividades ejecutadas, critico para I19. Los formatos 2024+ ya estan cubiertos
por los exports planos del pipeline actual.